# PREDICTIVE MODELING - DAILY PRACTICE EXAM PROBLEM

**Date**: December 30, 2025
**Topic**: Logistic Regression with ROC Curve Analysis and Model Diagnostics
**Difficulty Level**: Intermediate
**Estimated Duration**: 60-75 minutes
**Points**: 30 points total

## Problem Statement

You are working as a data scientist for a financial institution analyzing credit risk. The institution has historical data on 1'000 loan applicants with the following variables:

- **Response Variable (Y)**: `default` - Binary indicator (1 = defaulted, 0 = no default)
- **Predictor Variables**:
  - `income` - Annual income in thousands of dollars
  - `balance` - Current credit balance in dollars
  - `age` - Age of applicant in years
  - `employment_years` - Years at current employment
  - `credit_score` - Credit score (300-850 range)

Your task is to build a logistic regression model to predict loan default risk and evaluate its performance.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000

income = np.random.normal(50, 15, n).clip(10, 150)           # kCHF
balance = np.random.normal(1500, 700, n).clip(0, 5000)       # CHF
age = np.random.randint(18, 75, n)
employment_years = np.random.randint(0, 40, n)
credit_score = np.random.normal(680, 60, n).clip(300, 850)

# Linear predictor for default probability
lp = (
    -8.0
    + 0.02 * (income - 50)
    + 0.001 * (balance - 1500)
    - 0.01 * (credit_score - 680)
    + 0.03 * (employment_years - 10)
)
p_default = 1 / (1 + np.exp(-lp))
default = np.random.binomial(1, p_default)

df = pd.DataFrame({
    "default": default,
    "income": income,
    "balance": balance,
    "age": age,
    "employment_years": employment_years,
    "credit_score": credit_score,
})

df.to_csv("../data/loan_data.csv", index=False)

## Part A: Data Exploration and Model Fitting (10 points)

### A1. Load and Summarize Data (2 points)

```python
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

# Load the data
df = pd.read_csv('loan_data.csv')

# Your code here:
# 1. Display the first 5 rows and shape of the dataset
# 2. Check for missing values
# 3. Compute descriptive statistics for all variables
# 4. Cross-tabulate default frequencies
```

**Expected Output Format**:
- Shape: (1000, 6) with columns: id, default, income, balance, age, employment_years, credit_score
- Missing values: None or minimal
- Default distribution: Approximately 10% defaulters (class imbalance)

In [12]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

# Load the data
df = pd.read_csv('../data/loan_data.csv')

# Your code here:
print(df.head(5))
print(f"Shape: {df.shape}")
print(f"Contains any values: {df.isnull().values.any()}")
print("")
print("Descriptive Statistics:")
print(df.describe())

   default     income      balance  age  employment_years  credit_score
0        0  57.450712  2479.548806   47                26    673.717204
1        0  47.926035  2147.243578   48                34    517.791192
2        0  59.715328  1541.741259   55                20    691.208264
3        0  72.845448  1047.144256   28                24    616.694570
4        0  46.487699  1988.756320   52                27    763.431354
Shape: (1000, 6)
Contains any values: False

Descriptive Statistics:
           default       income      balance          age  employment_years  \
count  1000.000000  1000.000000  1000.000000  1000.000000       1000.000000   
mean      0.001000    50.299053  1554.127605    46.107000         19.755000   
std       0.031623    14.660762   686.636079    16.493441         11.475481   
min       0.000000    10.000000     0.000000    18.000000          0.000000   
25%       0.000000    40.286145  1075.630818    32.000000         10.000000   
50%       0.000000    50.

### A2. Model Fitting (3 points)

Fit a full logistic regression model using all predictor variables (income, balance, age, employment_years, credit_score).

```python
# Set random seed for reproducibility
np.random.seed(42)

# Create X (predictors) and y (response)
# Add constant term
# Fit GLM with binomial family using statsmodels

# Your code here:
# Fit logistic regression model
# Print summary statistics

# Questions:
# a) Which predictors are statistically significant at α = 0.05?
# b) Interpret the coefficient for 'balance' - what does a 1-unit increase mean
#    for the log-odds of default?
# c) What is the pseudo-R² (McFadden's R²) and model AIC?
```

In [13]:
import numpy as np
import statsmodels.api as sm

# Set random seed for reproducibility
np.random.seed(42)

# Create X (predictors) and y (response)
X = df[['income', 'balance', 'age', 'employment_years', 'credit_score']]
y = df['default']

# Add constant term
X_sm = sm.add_constant(X)

# Fit GLM with binomial family using statsmodels
model = sm.GLM(y, X_sm, family=sm.families.Binomial()).fit()

print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                default   No. Observations:                 1000
Model:                            GLM   Df Residuals:                      994
Model Family:                Binomial   Df Model:                            5
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -4.9158
Date:                Tue, 30 Dec 2025   Deviance:                       9.8317
Time:                        11:12:31   Pearson chi2:                     48.0
No. Iterations:                    12   Pseudo R-squ. (CS):           0.005965
Covariance Type:            nonrobust                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -1.9964     14.396  

#### a) Which predictors are statistically significant at α = 0.05?

A predictor is “significant” if: Its p‑value < 0.05 AND its 95% Conficende Ingerval does not include 0. Here, all p‑values are > 0.05 and all Conficende Intervals include 0, so we cannot reject $H_0$: $\beta_j = 0$ for any predictor.

#### b) Interpret the coefficient for 'balance' - what does a 1-unit increase mean for the log-odds of default?

It means that for every 1-unit increase in balance, the log-odds of default increases by 0.0022 (holding other predictors constant).

#### c) What is the pseudo-R² (McFadden's R²) and model AIC?

In [26]:
results = model._results

print(f"McFadden's R²: {results.pseudo_rsquared(kind='mcf')}")
print(f"AIC: {results.aic}")

McFadden's R²: 0.37831315515820385
AIC: 21.831672964165463


In [27]:
from scipy.stats import t
t.ppf(0.975, 95)

1.9852510034099262

### A3. Model Simplification via Hypothesis Test (5 points)

Test whether `employment_years` and `age` can be simultaneously removed from the model.

**Question**:
- State the null hypothesis (H₀) and alternative hypothesis (H₁)
- Which test statistic should you use?
- Implement the partial F-test (or likelihood ratio test) in Python
- At significance level α = 0.05, can we remove these variables?
- Show the calculation: If reduced model has Log-Likelihood = -245.3 and full model has LL = -241.2, what is your test statistic?

**Mathematical Background**:
The test statistic for comparing nested logistic models is:
$$\Lambda = 2(LL_{full} - LL_{reduced})$$

This statistic follows a χ² distribution with degrees of freedom equal to the number of restricted parameters.

## Part B: ROC Curve Analysis and Threshold Selection (10 points)

### B1. Create Train-Test Split (2 points)

```python
# Split data: 70% train, 30% test
X = df[['income', 'balance', 'age', 'employment_years', 'credit_score']]
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                      random_state=42, stratify=y)

# Your task:
# 1. Fit the final model on training data only
# 2. Generate probability predictions for test set
# 3. Verify the class distribution in train and test sets
```

### B2. Compute Confusion Matrix at Default Threshold (1.5 points)

Using threshold τ = 0.5, compute the confusion matrix on the test set.

**For a woman with**: income = $45K, balance = $1,500, age = 35, employment_years = 5, credit_score = 720

```python
# Predict probabilities at threshold 0.5
# Generate confusion matrix
# Calculate and interpret:
# - Sensitivity (Recall/True Positive Rate)
# - Specificity
# - Precision
# - False Positive Rate
# - F1 Score
```

**Formulas**:
- **Sensitivity** = TP / (TP + FN)
- **Specificity** = TN / (TN + FP)
- **Precision** = TP / (TP + FP)
- **FPR** = FP / (FP + TN)
- **F1** = 2 · (Precision · Recall) / (Precision + Recall)

### B3. Individual Prediction (1.5 points)

For the applicant described above, what is the predicted probability of default?

### B4. ROC Curve Construction (5 points)

**Task**: Construct the ROC curve by varying the classification threshold from 0 to 1.

```python
def compute_roc_curve(y_true, y_proba, thresholds=np.linspace(0, 1, 101)):
    """
    Compute ROC curve points (FPR, TPR) for varying thresholds.

    Parameters:
    - y_true: true binary labels
    - y_proba: predicted probabilities
    - thresholds: array of threshold values

    Returns:
    - fpr: false positive rates
    - tpr: true positive rates (sensitivities)
    """
    # Your implementation here

    # Plot ROC curve
    # Calculate AUC (Area Under Curve)
    # Interpret the result
```

**Questions**:
1. Plot the ROC curve and the diagonal line (random classifier)
2. Calculate the Area Under the Curve (AUC) using the trapezoidal rule
3. Interpret the AUC value - what does it tell you about model performance?
4. At what threshold(s) do you achieve:
   - 90% sensitivity?
   - 80% specificity?
5. For this financial institution, which threshold would you recommend? Justify your choice considering both false positives (incorrectly denying loans) and false negatives (incorrectly approving defaults)

---

## Part C: Model Diagnostics and Assumptions (10 points)

### C1. Residual Analysis (5 points)

For logistic regression, Pearson residuals and deviance residuals are key diagnostics.

**Pearson Residuals**:
$$r_i^{Pearson} = \frac{y_i - \hat{p}_i}{\sqrt{\hat{p}_i(1-\hat{p}_i)}}$$

**Deviance Residuals**:
$$r_i^{deviance} = \text{sign}(y_i - \hat{p}_i) \sqrt{-2[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)]}$$

```python
# Calculate residuals from your fitted model
# Create diagnostic plots:
# 1. Residuals vs Fitted probabilities (look for patterns indicating lack of fit)
# 2. Histogram of residuals (should be roughly symmetric around 0)
# 3. Q-Q plot (assess normality of residuals - note: for logistic regression,
#    normality is less critical than for linear regression)

# Question: Do the diagnostic plots suggest any violations of model assumptions?
#          Are there any problematic predictions?
```

### C2. Multicollinearity Check (3 points)

```python
# Calculate Variance Inflation Factor (VIF) for each predictor
# VIF(X_j) = 1 / (1 - R²_j)
# where R²_j is from regressing X_j on other X variables

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Your code:
# 1. Calculate VIF for all predictors
# 2. Identify predictors with VIF > 10 (indicating problematic multicollinearity)
# 3. Interpret results

# VIF Interpretation:
# VIF = 1: No correlation
# VIF = 1-5: Moderate but acceptable correlation
# VIF > 5: Concerning, consider variable reduction
```

### C3. Goodness of Fit (2 points)

**Hosmer-Lemeshow Test**: Tests whether predicted probabilities match observed outcomes in bins.

- Null hypothesis: Model fits the data well
- The test statistic follows χ² distribution with k-2 degrees of freedom (k = number of bins, typically 10)

**Your task**:
1. Briefly explain what this test assesses
2. Conceptually describe how you would implement it (bin predictions into deciles, compare observed vs expected counts)
3. If p-value = 0.23, what conclusion do you draw?

---

## Solution Guide

### Part A Solutions

**A1. Data Summary**:
The dataset contains 1,000 loan applicants. The default variable has approximately 100 defaulters (10%) and 900 non-defaulters (class imbalance of 9:1).

**A2. Model Output Interpretation**:

Typical coefficient interpretation from GLM summary:
```
Coefficients:
                    coef    std err      z      P>|z|      [95% Conf. Int.]
const             -8.5120      1.234   -6.90      0.000     -11.03  -6.00
income             0.0045      0.0012    3.75      0.000       0.002   0.007
balance            0.0008      0.0002    4.00      0.000       0.0004  0.0012
age               -0.0234      0.0156   -1.50      0.134      -0.054   0.007
employment_years   0.1234      0.0567    2.18      0.029       0.012   0.235
credit_score      -0.0089      0.0021   -4.24      0.000      -0.013  -0.005
```

**Significant predictors** (α = 0.05): income, balance, employment_years, credit_score
**Not significant**: age (p = 0.134)

**Balance Interpretation**: A 1-unit increase (1 dollar) in balance increases the log-odds of default by 0.0008, or approximately 0.08% per dollar. For a $1,000 increase, log-odds increase by 0.8.

**A3. Hypothesis Test**:

H₀: Coefficients for age and employment_years both equal zero (restricted model)
H₁: At least one coefficient ≠ 0

Test statistic: Λ = 2(-245.3 - (-241.2)) = 2(4.1) = 8.2
df = 2, χ²₂,₀.₀₅ = 5.99

Since 8.2 > 5.99, **reject H₀** at α = 0.05. We cannot remove both variables simultaneously.

---

### Part B Solutions

**B1. Split Verification**:
- Train set: 700 samples (100 defaulters, 600 non-defaulters)
- Test set: 300 samples (50 defaulters, 250 non-defaulters)

**B2. Confusion Matrix Example** (at threshold 0.5):
```
                Predicted No Default    Predicted Default
Actual No Default      240                  10
Actual Default          35                  15
```

Metrics:
- Sensitivity: 15 / (15 + 35) = 0.30 (30%)
- Specificity: 240 / (240 + 10) = 0.96 (96%)
- Precision: 15 / (15 + 10) = 0.60 (60%)
- FPR: 10 / (10 + 240) = 0.04 (4%)
- F1: 2 · (0.60 · 0.30) / (0.60 + 0.30) = 0.40

**B3. Individual Prediction**:
Log-odds = -8.512 + 0.0045(45) + 0.0008(1500) - 0.0234(35) + 0.1234(5) - 0.0089(720)
         = -8.512 + 0.203 + 1.2 - 0.819 + 0.617 - 6.408
         = -13.719

Probability = e^(-13.719) / (1 + e^(-13.719)) ≈ 0.00001 (essentially 0% risk)

**B4. ROC Curve**:
- At threshold 0 (classify all as default): TPR = 1.0, FPR = 1.0 (top-right)
- At threshold 1 (classify all as no default): TPR = 0.0, FPR = 0.0 (origin)
- AUC typically ranges 0.75-0.85 for this type of credit risk model
- A useful threshold balancing false positives and negatives might be τ = 0.20-0.30

**Recommendation**: In lending, the cost of false negatives (bad loans approved) typically exceeds false positives (good loans denied). A lower threshold (0.25) might capture more defaulters despite some false alarms.

---

### Part C Solutions

**C1. Diagnostic Interpretation**:
For logistic regression, patterns in residual plots indicate:
- Systematic deviation from 0 line: lack of fit
- Outliers beyond ±2-3: potential misclassified observations
- Heteroscedasticity pattern: less concerning than linear regression

**C2. VIF Results**:
Expected VIFs:
- income: 2.1 (acceptable)
- balance: 1.8 (acceptable)
- age: 3.4 (acceptable)
- employment_years: 2.6 (acceptable)
- credit_score: 2.9 (acceptable)

All below 5, indicating **no problematic multicollinearity**.

**C3. Hosmer-Lemeshow Test**:
p-value = 0.23 > 0.05 → **Fail to reject H₀**. Model fits adequately.

---

## Key Learning Objectives Tested

✓ Logistic regression model fitting and interpretation
✓ Hypothesis testing for nested models
✓ Classification performance metrics (confusion matrix, ROC curves)
✓ Threshold selection and its practical implications
✓ Model diagnostics and assumptions
✓ Multicollinearity detection
✓ Goodness-of-fit assessment

---

## References

1. **Series 5 - Logistic Regression**: Exercises 5.1-5.6 covering odds, probabilities, model interpretation
2. **Exam FS18 - Problem 4**: Pima Indians diabetes prediction with logistic regression
3. **Course Slides 05_slides_LR.pdf**: Classification fundamentals and logistic regression
4. **Main textbook (ISLR)**: Chapter 4.3 on Logistic Regression and classification metrics

---

## Appendix: Python Code Template

```python
# Complete solution template
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import classification_report

# 1. Load and explore
df = pd.read_csv('loan_data.csv')
print(df.shape)
print(df.info())
print(df.describe())

# 2. Prepare data
X = df[['income', 'balance', 'age', 'employment_years', 'credit_score']]
y = df['default']
X = sm.add_constant(X)

# 3. Fit full model
model_full = sm.GLM(y, X, family=sm.families.Binomial())
results_full = model_full.fit()
print(results_full.summary())

# 4. ROC curve
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                      random_state=42, stratify=y)
model = sm.GLM(y_train, X_train, family=sm.families.Binomial()).fit()
y_proba = model.predict(X_test)

fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.title('ROC Curve - Loan Default Prediction')
plt.show()

# 5. Confusion matrix at custom threshold
threshold = 0.3
y_pred = (y_proba >= threshold).astype(int)
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix (threshold={threshold}):")
print(cm)
print(classification_report(y_test, y_pred))
```

---

**Total Points: 30**
**Exam Format**: Written answers + Python code on computer (as per course requirements)
